In [ ]:
!uv pip install -q gdown
!gdown --id 1PojPVpXGBAqzHQi97QAFhJ9gnPsXxveS -O dataset.zip
!unzip -q dataset.zip

/Users/bottleofjuice/Documents/hse/ai360/AdaptiveTemperatureRecSys/.venv/lib/python3.13/site-packages/gdown/__main__.py:139: FutureWarning: Option `--id` was deprecated in version 4.3.1 and will be removed in 5.0. You don't need to pass it anymore to use a file ID.
  warnings.warn(
Downloading...
From (original): https://drive.google.com/uc?id=1PojPVpXGBAqzHQi97QAFhJ9gnPsXxveS
From (redirected): https://drive.google.com/uc?id=1PojPVpXGBAqzHQi97QAFhJ9gnPsXxveS&confirm=t&uuid=152a3b64-6563-47d2-87c6-81e41390272c
To: /Users/bottleofjuice/Documents/hse/ai360/AdaptiveTemperatureRecSys/dataset.zip
 68%|███████████████████████████▎            | 243M/356M [01:16<00:43, 2.61MB/s]^C
Traceback (most recent call last):
  File "/Users/bottleofjuice/Documents/hse/ai360/AdaptiveTemperatureRecSys/.venv/bin/gdown", line 10, in <module>
    sys.exit(main())
             ~~~~^^
  File "/Users/bottleofjuice/Documents/hse/ai360/AdaptiveTemperatureRecSys/.venv/lib/python3.13/site-packages/gdown/__main__.py"

In [ ]:
import torch 
import torch.nn as nn
import torch.nn.functional as F
import polars as pl
import comet_ml

from config import MAX_TRAIN_EVENTS_PER_USER, MAX_LEN, COMET_API_KEY
from dataset import (get_general_data, get_item_to_token, get_item_to_freq,
                      get_train_events, get_train_histories, TrainingDataset, 
                      TestDataset, get_test_histories)
from model import Graph
from train import train_loop
from eval import eval_loop

In [ ]:
train, test, embeddings, artists, test_targets = get_general_data()
item_to_token = get_item_to_token(train, vocab_size=None)
item_to_freq = get_item_to_freq(train)

train_events = get_train_events(train, item_to_token, item_to_freq=item_to_freq)
train_histories = get_train_histories(train_events)

test_histories = get_test_histories(test, train_events)

In [ ]:
vocab_size = item_to_token['token_id'].max() + 1

In [ ]:
dataloader = TrainingDataset(train_histories, batch_size=32,
                             seq_len=MAX_TRAIN_EVENTS_PER_USER,
                             shuffle=True, device='cuda',
                             uniform_negative_items=30_000,
                             in_batch_negative_items=0,
                             vocab_size=vocab_size)

In [ ]:
NUM_EPOCH = 5

TAU_MAX = 0.08
TAU_MIN = 0.04

In [ ]:
from dataset import TrainingBatch
import numpy as np

tokens_in_epoch = dataloader.total_num_tokens

def get_tau(net: Graph,
            pos_logits: torch.Tensor,
            neg_logits: torch.Tensor,
            batch: TrainingBatch) -> float:

    epoch = net.tokens_passed / tokens_in_epoch

    return float(TAU_MIN + np.cos(epoch / NUM_EPOCH * np.pi / 2) * (TAU_MAX - TAU_MIN))




In [ ]:
experiment = comet_ml.Experiment(
    api_key=COMET_API_KEY,
    project_name="AdaptiveTemperature",
    workspace="maksim-bessolitsyn",
)

graph = Graph(
    vocab_size=vocab_size,
    max_seq_len=MAX_LEN,
    n_layers=4,
    dropout=0.1,
    tau=get_tau,
    log_q_correction=1,
    is_cosine_similarity=True,
).cuda()
compiled_graph = torch.compile(graph, mode='default')

optimizer = torch.optim.AdamW(
    compiled_graph.parameters(),
    lr=1e-3,
    weight_decay=1e-4,
    fused=True
)

train_loop(
    compiled_graph,
    num_epochs=5,
    train_dataloader=dataloader,
    optimizer=optimizer,
    grad_clip=10.0,
    grad_accum_steps=1,
    writer=experiment,
)

result = eval_loop(
    compiled_graph,
    TestDataset(test_histories, batch_size=128),
    test_histories=test_histories,
    test_targets=test_targets,
    item_to_token=item_to_token,
    writer=experiment,
)

experiment.end()

result

COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO: Comet.ml Experiment Summary
COMET INFO: ---------------------------------------------------------------------------------------
COMET INFO:   Data:
COMET INFO:     display_summary_level : 1
COMET INFO:     name                  : balanced_bovid_5877
COMET INFO:     url                   : https://www.comet.com/maksim-bessolitsyn/adaptivetemperature/c35e13bf7f2a4febaf6212fc3e4773cc
COMET INFO:   Metrics [count] (min, max):
COMET INFO:     loss                           : 31.320039749145508
COMET INFO:     optim/grad_norm [3679]         : (0.06377652287483215, 98.14552307128906)
COMET INFO:     time/tokens_per_sec(k) [3679]  : (3.0, 21.0)
COMET INFO:     time/train_step_time(s) [3679] : (0.14747142791748047, 0.8659729957580566)
COMET INFO:     train/loss [3680]              : (29.259286880493164, 31.38616180419922)
COMET INFO:     train/neg_logit_max [3680]     : (0.3125, 0.898

{'hitrate': 0.2848635368263633,
 'recall': 0.08821911288496917,
 'ndcg': 0.03432448724991838,
 'coverage': 0.08476873445026312}